In [ ]:
import os, sys
from scrapxd import Scrapxd

parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))

if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

csv_path = os.path.join(parent_dir, "data", "film_slugs.csv")

TOTAL_PAGES = 667

client = Scrapxd()
offset = 0
limit = 72

for page in range(1, TOTAL_PAGES + 1):
    print(f"Scraping page {page}/{TOTAL_PAGES}...")
    data = client.search_films(offset=offset, limit=limit)
    slugs = [film.slug for film in data.films]

    if slugs:
        with open(csv_path, "a") as f:
            for i in range(len(slugs)):
                f.write(f"{i + offset};{slugs[i]}\n")

    offset += limit

In [ ]:
import os, sys

parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))

if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

csv_path = os.path.join(parent_dir, "data", "film_slugs.csv")

In [ ]:
import pandas as pd

df = pd.read_csv(csv_path, sep=";", names=["rank", "slug"])
df["rank"] = df["rank"].astype(int) + 1

df.to_csv(csv_path, sep=";", index=False)

In [ ]:
from scrapxd import Scrapxd


client = Scrapxd()

data_csv_path = os.path.join(parent_dir, "data", "films_data.csv")

slugs = df["slug"].tolist()

for i in range(0, 48000):
    if (i + 1) % 100 == 0:
        print(f"Fetching film {i + 1}/{len(slugs)}: {slugs[i]}")

    film = client.get_film(slugs[i])

    data = [i+1, film.id, film.slug, film.title, film.original_title, film.year, 
            film.runtime, film.director, film.genre, film.country, film.language, film.cast, 
            film.actors, film.crew, film.studio, film.synopsis, film.tagline, film.themes, 
            film.alternative_titles, film.avg_rating, film.total_logs, film.poster]
    
    data_str = ";".join([str(item).replace(";", ",") for item in data]) 

    with open(data_csv_path, "a") as f:
        f.write(f"{data_str}\n")

In [13]:
df = pd.read_csv(data_csv_path, sep=";", on_bad_lines='skip')
df.sort_values(by='total_logs', ascending=False, inplace=True)
df.reset_index(drop=True, inplace=True)
df['rank'] = df.index + 1
df.drop(columns=['cast'], inplace=True)
df = df[df.index < 48000]
df.to_csv(data_csv_path, sep=";", index=False)